# Session 9 - Sorting & Counting with CUDA
## CT225H - Parallel Programming with CUDA
### Nguyễn Thanh Trọng - B2305615

**Bài toán 1:** Đếm số lần xuất hiện của mỗi số trong mảng  
**Bài toán 2:** Counting Sort  
**Bài toán 3:** Radix Sort (LSD, base 256)

In [1]:
# Kiểm tra GPU trên Colab
!nvidia-smi

Tue Mar 24 02:02:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Bài toán 1: Đếm số lần xuất hiện (Histogram)

Cho mảng $N = 10{,}000{,}000$ số nguyên trong phạm vi $[0, 999]$.  
Đếm số lần xuất hiện của mỗi giá trị.

**Phương pháp GPU:**
1. Mỗi thread xử lý nhiều phần tử qua grid-stride loop
2. Sử dụng **Shared Memory histogram** cục bộ mỗi block để giảm áp lực `atomicAdd` trên global memory
3. Cuối mỗi block, cộng dồn histogram cục bộ vào histogram global bằng `atomicAdd`

In [2]:
%%writefile counting_occurrences.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>

#define N 10000000
#define MAX_VAL 1000
#define BLOCK_SIZE 256

// ===================== CPU Version =====================
void histogramCPU(int *data, int *hist, int n, int maxVal) {
    for (int i = 0; i < maxVal; i++) hist[i] = 0;
    for (int i = 0; i < n; i++) hist[data[i]]++;
}

// ===================== GPU Kernel =====================
// Sử dụng shared memory histogram cục bộ mỗi block
// để giảm contention trên global memory atomicAdd
__global__ void histogramKernel(int *data, int *hist, int n, int maxVal) {
    // Shared memory histogram cục bộ cho block này
    extern __shared__ int sharedHist[];

    // Khởi tạo shared histogram = 0
    for (int i = threadIdx.x; i < maxVal; i += blockDim.x)
        sharedHist[i] = 0;
    __syncthreads();

    // Mỗi thread đếm vào shared histogram (grid-stride loop)
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) {
        atomicAdd(&sharedHist[data[i]], 1);
    }
    __syncthreads();

    // Cộng shared histogram vào global histogram
    for (int i = threadIdx.x; i < maxVal; i += blockDim.x) {
        if (sharedHist[i] > 0)
            atomicAdd(&hist[i], sharedHist[i]);
    }
}

int verify(int *ref, int *result, int n) {
    for (int i = 0; i < n; i++)
        if (ref[i] != result[i]) {
            printf("Mismatch at %d: CPU=%d, GPU=%d\n", i, ref[i], result[i]);
            return 0;
        }
    return 1;
}

int main() {
    int *h_data, *h_hist_cpu, *h_hist_gpu;
    int *d_data, *d_hist;

    size_t dataBytes = N * sizeof(int);
    size_t histBytes = MAX_VAL * sizeof(int);

    h_data = (int *)malloc(dataBytes);
    h_hist_cpu = (int *)malloc(histBytes);
    h_hist_gpu = (int *)malloc(histBytes);

    srand(42);
    for (int i = 0; i < N; i++)
        h_data[i] = rand() % MAX_VAL;

    // ==================== CPU ====================
    clock_t cpu_start = clock();
    histogramCPU(h_data, h_hist_cpu, N, MAX_VAL);
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;

    printf("=== Counting Occurrences (N = %d, Range = [0, %d)) ===\n", N, MAX_VAL);
    printf("[CPU] Time: %.4f ms\n", cpu_time);

    // ==================== GPU ====================
    cudaMalloc(&d_data, dataBytes);
    cudaMalloc(&d_hist, histBytes);
    cudaMemcpy(d_data, h_data, dataBytes, cudaMemcpyHostToDevice);
    cudaMemset(d_hist, 0, histBytes);

    int numBlocks = (N + BLOCK_SIZE - 1) / BLOCK_SIZE;
    // Giới hạn số block để tối ưu
    if (numBlocks > 1024) numBlocks = 1024;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    histogramKernel<<<numBlocks, BLOCK_SIZE, MAX_VAL * sizeof(int)>>>(d_data, d_hist, N, MAX_VAL);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(h_hist_gpu, d_hist, histBytes, cudaMemcpyDeviceToHost);

    printf("[GPU] Time: %.4f ms\n", gpu_time);
    printf("Speedup: %.2fx\n", cpu_time / gpu_time);
    printf("Correct: %s\n", verify(h_hist_cpu, h_hist_gpu, MAX_VAL) ? "Yes" : "No");

    // In một vài giá trị để kiểm tra
    printf("\n--- Sample histogram (first 10 values) ---\n");
    for (int i = 0; i < 10; i++)
        printf("  Value %d: CPU=%d, GPU=%d\n", i, h_hist_cpu[i], h_hist_gpu[i]);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_data);
    cudaFree(d_hist);
    free(h_data);
    free(h_hist_cpu);
    free(h_hist_gpu);

    return 0;
}

Writing counting_occurrences.cu


In [3]:
# Biên dịch và chạy Counting Occurrences
!nvcc -o counting_occurrences counting_occurrences.cu -arch=sm_75
!./counting_occurrences

=== Counting Occurrences (N = 10000000, Range = [0, 1000)) ===
[CPU] Time: 30.7730 ms
[GPU] Time: 0.3142 ms
Speedup: 97.94x
Correct: Yes

--- Sample histogram (first 10 values) ---
  Value 0: CPU=10113, GPU=10113
  Value 1: CPU=10087, GPU=10087
  Value 2: CPU=9990, GPU=9990
  Value 3: CPU=10083, GPU=10083
  Value 4: CPU=9979, GPU=9979
  Value 5: CPU=10071, GPU=10071
  Value 6: CPU=10079, GPU=10079
  Value 7: CPU=9899, GPU=9899
  Value 8: CPU=10211, GPU=10211
  Value 9: CPU=10088, GPU=10088


## Bài toán 2: Counting Sort

Sắp xếp mảng $N = 10{,}000{,}000$ số nguyên trong phạm vi $[0, 999]$ bằng Counting Sort.

**Phương pháp GPU:**
1. **Bước 1 – Histogram:** Kernel song song đếm tần suất (giống bài toán 1)
2. **Bước 2 – Prefix Sum:** Tính prefix sum trên histogram (Blelloch scan hoặc tuần tự do K nhỏ)
3. **Bước 3 – Scatter:** Kernel song song đặt phần tử vào vị trí đúng dựa trên prefix sum

In [10]:
%%writefile counting_sort.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>
#include <string.h>

#define N 10000000
#define MAX_VAL 1000
#define BLOCK_SIZE 256

// ===================== CPU Counting Sort =====================
void countingSortCPU(int *data, int *output, int n, int maxVal) {
    int *count = (int *)calloc(maxVal, sizeof(int));

    // Đếm
    for (int i = 0; i < n; i++)
        count[data[i]]++;

    // Prefix sum
    for (int i = 1; i < maxVal; i++)
        count[i] += count[i - 1];

    // Placement (stable, duyệt ngược)
    for (int i = n - 1; i >= 0; i--) {
        output[count[data[i]] - 1] = data[i];
        count[data[i]]--;
    }

    free(count);
}

// ===================== GPU Histogram Kernel =====================
__global__ void histogramKernel(int *data, int *hist, int n, int maxVal) {
    extern __shared__ int sharedHist[];
    for (int i = threadIdx.x; i < maxVal; i += blockDim.x)
        sharedHist[i] = 0;
    __syncthreads();

    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride)
        atomicAdd(&sharedHist[data[i]], 1);
    __syncthreads();

    for (int i = threadIdx.x; i < maxVal; i += blockDim.x)
        if (sharedHist[i] > 0)
            atomicAdd(&hist[i], sharedHist[i]);
}

// ===================== GPU Prefix Sum (tuần tự trên CPU vì K nhỏ) =====================
// Với K = 1000, prefix sum trên CPU nhanh hơn launch kernel

// ===================== GPU Scatter Kernel =====================
// Mỗi thread xử lý một giá trị v, đặt count[v] phần tử = v vào output
__global__ void scatterKernel(int *output, int *prefixSum, int *count, int maxVal) {
    int v = blockIdx.x * blockDim.x + threadIdx.x;
    if (v < maxVal) {
        int start = (v == 0) ? 0 : prefixSum[v - 1];
        int end = prefixSum[v];
        for (int i = start; i < end; i++)
            output[i] = v;
    }
}

int verify(int *ref, int *result, int n) {
    for (int i = 0; i < n; i++)
        if (ref[i] != result[i]) {
            printf("Mismatch at %d: CPU=%d, GPU=%d\n", i, ref[i], result[i]);
            return 0;
        }
    return 1;
}

int main() {
    int *h_data, *h_sorted_cpu, *h_sorted_gpu;
    int *d_data, *d_hist, *d_prefix, *d_output;

    size_t dataBytes = N * sizeof(int);
    size_t histBytes = MAX_VAL * sizeof(int);

    h_data = (int *)malloc(dataBytes);
    h_sorted_cpu = (int *)malloc(dataBytes);
    h_sorted_gpu = (int *)malloc(dataBytes);
    int *h_hist = (int *)malloc(histBytes);
    int *h_prefix = (int *)malloc(histBytes);

    srand(42);
    for (int i = 0; i < N; i++)
        h_data[i] = rand() % MAX_VAL;

    // ==================== CPU Counting Sort ====================
    clock_t cpu_start = clock();
    countingSortCPU(h_data, h_sorted_cpu, N, MAX_VAL);
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;

    printf("=== Counting Sort (N = %d, Range = [0, %d)) ===\n", N, MAX_VAL);
    printf("[CPU] Time: %.4f ms\n", cpu_time);

    // ==================== GPU Counting Sort ====================
    cudaMalloc(&d_data, dataBytes);
    cudaMalloc(&d_hist, histBytes);
    cudaMalloc(&d_prefix, histBytes);
    cudaMalloc(&d_output, dataBytes);

    cudaMemcpy(d_data, h_data, dataBytes, cudaMemcpyHostToDevice);
    cudaMemset(d_hist, 0, histBytes);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    int histBlocks = min(1024, (N + BLOCK_SIZE - 1) / BLOCK_SIZE);

    cudaEventRecord(start);

    // Bước 1: Histogram trên GPU
    histogramKernel<<<histBlocks, BLOCK_SIZE, MAX_VAL * sizeof(int)>>>(d_data, d_hist, N, MAX_VAL);
    cudaDeviceSynchronize();

    // Bước 2: Prefix sum (trên CPU vì K = 1000 rất nhỏ)
    cudaMemcpy(h_hist, d_hist, histBytes, cudaMemcpyDeviceToHost);
    h_prefix[0] = h_hist[0];
    for (int i = 1; i < MAX_VAL; i++)
        h_prefix[i] = h_prefix[i - 1] + h_hist[i];
    cudaMemcpy(d_prefix, h_prefix, histBytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_hist, h_hist, histBytes, cudaMemcpyHostToDevice);

    // Bước 3: Scatter trên GPU
    int scatterBlocks = (MAX_VAL + BLOCK_SIZE - 1) / BLOCK_SIZE;
    scatterKernel<<<scatterBlocks, BLOCK_SIZE>>>(d_output, d_prefix, d_hist, MAX_VAL);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time, start, stop);

    cudaMemcpy(h_sorted_gpu, d_output, dataBytes, cudaMemcpyDeviceToHost);

    printf("[GPU] Time: %.4f ms (incl. CPU prefix sum)\n", gpu_time);
    printf("Speedup: %.2fx\n", cpu_time / gpu_time);
    printf("Correct: %s\n", verify(h_sorted_cpu, h_sorted_gpu, N) ? "Yes" : "No");

    // In mẫu
    printf("\n--- Sample sorted output (first 20) ---\n");
    for (int i = 0; i < 20; i++)
        printf("%d ", h_sorted_gpu[i]);
    printf("\n");

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_data);
    cudaFree(d_hist);
    cudaFree(d_prefix);
    cudaFree(d_output);
    free(h_data);
    free(h_sorted_cpu);
    free(h_sorted_gpu);
    free(h_hist);
    free(h_prefix);

    return 0;
}

Overwriting counting_sort.cu


In [6]:
# Biên dịch và chạy Counting Sort
!nvcc -o counting_sort counting_sort.cu -arch=sm_75
!./counting_sort

=== Counting Sort (N = 10000000, Range = [0, 1000)) ===
[CPU] Time: 117.8440 ms
[GPU] Time: 5.1651 ms (incl. CPU prefix sum)
Speedup: 22.82x
Correct: Yes

--- Sample sorted output (first 20) ---
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 


## Bài toán 3: Radix Sort (LSD, base 256)

Sắp xếp mảng $N = 10{,}000{,}000$ số nguyên không âm bằng Radix Sort.

**Phương pháp GPU (LSD Radix Sort):**
- Cơ số 256 → 4 pass cho số 32-bit
- Mỗi pass:
  1. **Histogram:** Kernel song song đếm tần suất digit hiện tại
  2. **Prefix Sum:** Trên CPU (vì K=256 rất nhỏ)
  3. **Scatter:** Tuần tự trên CPU để đảm bảo stable sort (hoặc dùng kỹ thuật per-block offset)

In [8]:
%%writefile radix_sort.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <time.h>
#include <string.h>

#define N 10000000
#define MAX_VAL 1000
#define BLOCK_SIZE 256
#define RADIX 256
#define NUM_PASSES 4

// ===================== CPU Radix Sort (LSD, base 256) =====================
void radixSortCPU(int *input, int *output, int n) {
    int *src = input;
    int *dst = output;

    for (int pass = 0; pass < NUM_PASSES; pass++) {
        int shift = pass * 8;
        int count[RADIX] = {0};

        for (int i = 0; i < n; i++)
            count[(src[i] >> shift) & 0xFF]++;

        for (int i = 1; i < RADIX; i++)
            count[i] += count[i - 1];

        for (int i = n - 1; i >= 0; i--) {
            int digit = (src[i] >> shift) & 0xFF;
            dst[count[digit] - 1] = src[i];
            count[digit]--;
        }

        int *temp = src;
        src = dst;
        dst = temp;
    }

    // Nếu NUM_PASSES chẵn, kết quả nằm trong src (== input ban đầu)
    // Nếu lẻ, cần copy
    if (NUM_PASSES % 2 == 1)
        memcpy(input, output, n * sizeof(int));
}

// ===================== GPU Histogram Kernel per digit =====================
__global__ void radixHistogramKernel(int *data, int *hist, int n, int shift) {
    __shared__ int sharedHist[RADIX];

    // Khởi tạo shared histogram
    if (threadIdx.x < RADIX)
        sharedHist[threadIdx.x] = 0;
    __syncthreads();

    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < n; i += stride) {
        int digit = (data[i] >> shift) & 0xFF;
        atomicAdd(&sharedHist[digit], 1);
    }
    __syncthreads();

    // Cộng vào global histogram
    if (threadIdx.x < RADIX)
        atomicAdd(&hist[threadIdx.x], sharedHist[threadIdx.x]);
}

int verify(int *ref, int *result, int n) {
    for (int i = 0; i < n; i++)
        if (ref[i] != result[i]) {
            printf("Mismatch at %d: ref=%d, result=%d\n", i, ref[i], result[i]);
            return 0;
        }
    return 1;
}

int main() {
    int *h_data, *h_temp, *h_sorted_cpu, *h_sorted_gpu;
    int *d_data, *d_hist;

    size_t dataBytes = N * sizeof(int);
    size_t histBytes = RADIX * sizeof(int);

    h_data = (int *)malloc(dataBytes);
    h_temp = (int *)malloc(dataBytes);
    h_sorted_cpu = (int *)malloc(dataBytes);
    h_sorted_gpu = (int *)malloc(dataBytes);

    srand(42);
    for (int i = 0; i < N; i++)
        h_data[i] = rand() % MAX_VAL;

    // ==================== CPU Radix Sort ====================
    memcpy(h_sorted_cpu, h_data, dataBytes);
    memcpy(h_temp, h_data, dataBytes);

    clock_t cpu_start = clock();
    radixSortCPU(h_sorted_cpu, h_temp, N);
    clock_t cpu_end = clock();
    double cpu_time = (double)(cpu_end - cpu_start) / CLOCKS_PER_SEC * 1000.0;

    printf("=== Radix Sort LSD base-256 (N = %d) ===\n", N);
    printf("[CPU] Time: %.4f ms\n", cpu_time);

    // ==================== GPU-assisted Radix Sort ====================
    // GPU thực hiện histogram, CPU thực hiện prefix sum + scatter
    memcpy(h_sorted_gpu, h_data, dataBytes);
    int *h_src = h_sorted_gpu;
    int *h_dst = h_temp;

    cudaMalloc(&d_data, dataBytes);
    cudaMalloc(&d_hist, histBytes);

    int numBlocks = min(1024, (N + BLOCK_SIZE - 1) / BLOCK_SIZE);
    int h_hist[RADIX];

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    for (int pass = 0; pass < NUM_PASSES; pass++) {
        int shift = pass * 8;

        // Copy dữ liệu lên GPU
        cudaMemcpy(d_data, h_src, dataBytes, cudaMemcpyHostToDevice);
        cudaMemset(d_hist, 0, histBytes);

        // Histogram trên GPU
        radixHistogramKernel<<<numBlocks, BLOCK_SIZE>>>(d_data, d_hist, N, shift);
        cudaDeviceSynchronize();

        // Copy histogram về CPU
        cudaMemcpy(h_hist, d_hist, histBytes, cudaMemcpyDeviceToHost);

        // Prefix sum trên CPU
        for (int i = 1; i < RADIX; i++)
            h_hist[i] += h_hist[i - 1];

        // Scatter trên CPU (đảm bảo stable)
        for (int i = N - 1; i >= 0; i--) {
            int digit = (h_src[i] >> shift) & 0xFF;
            h_dst[h_hist[digit] - 1] = h_src[i];
            h_hist[digit]--;
        }

        // Swap
        int *tmp = h_src;
        h_src = h_dst;
        h_dst = tmp;
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpu_time;
    cudaEventElapsedTime(&gpu_time, start, stop);

    printf("[GPU-assisted] Time: %.4f ms (GPU histogram + CPU prefix/scatter)\n", gpu_time);
    printf("Speedup: %.2fx\n", cpu_time / gpu_time);
    printf("Correct: %s\n", verify(h_sorted_cpu, h_src, N) ? "Yes" : "No");

    // In mẫu
    printf("\n--- Sample sorted output (first 20) ---\n");
    for (int i = 0; i < 20; i++)
        printf("%d ", h_src[i]);
    printf("\n");

    printf("--- Sample sorted output (last 20) ---\n");
    for (int i = N - 20; i < N; i++)
        printf("%d ", h_src[i]);
    printf("\n");

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(d_data);
    cudaFree(d_hist);
    free(h_data);
    free(h_temp);
    free(h_sorted_cpu);
    free(h_sorted_gpu);

    return 0;
}

Writing radix_sort.cu


In [9]:
# Biên dịch và chạy Radix Sort
!nvcc -o radix_sort radix_sort.cu -arch=sm_75
!./radix_sort

=== Radix Sort LSD base-256 (N = 10000000) ===
[CPU] Time: 532.9330 ms
[GPU-assisted] Time: 376.8464 ms (GPU histogram + CPU prefix/scatter)
Speedup: 1.41x
Correct: Yes

--- Sample sorted output (first 20) ---
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 
--- Sample sorted output (last 20) ---
999 999 999 999 999 999 999 999 999 999 999 999 999 999 999 999 999 999 999 999 
